# Statistical tests: frequentist, Bayesian, and sequential

This notebook runs three different analysis approaches on the same
A/B test data and compares their conclusions. Each approach offers
different strengths: frequentist provides error-rate guarantees,
Bayesian gives direct probability statements, and sequential testing
enables valid early stopping.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import sys
sys.path.insert(0, '..')

from src.experiment import (
    frequentist_test, bayesian_ab, sequential_test, summary_report, print_summary
)
from src.visualizations import (
    plot_conversion_bar, plot_posterior_distributions,
    plot_sequential_monitoring
)

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

## 1. Load experiment data

In [ ]:
df = pd.read_csv('../data/ab_test_results.csv')
print(f'Total records: {len(df):,}')
print(f'Experiments: {df["experiment_id"].unique()}')

# Summary per experiment
for exp_id in df['experiment_id'].unique():
    exp = df[df['experiment_id'] == exp_id]
    for var in ['control', 'treatment']:
        subset = exp[exp['variant'] == var]
        rate = subset['converted'].mean()
        print(f'  {exp_id} / {var}: n={len(subset):,}, conversion={rate:.3%}')

## 2. Select experiment for deep analysis

We use exp_001 (website redesign) as the primary example, then
apply all three methods to each experiment.

In [ ]:
exp1 = df[df['experiment_id'] == 'exp_001']
control = exp1[exp1['variant'] == 'control']
treatment = exp1[exp1['variant'] == 'treatment']

print(f'Experiment 001: Website redesign')
print(f'  Control:   n={len(control):,}, conversion={control["converted"].mean():.3%}')
print(f'  Treatment: n={len(treatment):,}, conversion={treatment["converted"].mean():.3%}')
print(f'  Observed lift: {treatment["converted"].mean() - control["converted"].mean():+.3%} (absolute)')

## 3. Frequentist z-test

The frequentist approach tests the null hypothesis that the
conversion rates are equal. It provides a p-value and confidence
interval for the difference in proportions.

In [ ]:
freq_result = frequentist_test(
    control['converted'].values,
    treatment['converted'].values,
    metric='conversion',
    alpha=0.05
)

print('Frequentist z-test results (exp_001):')
print(f'  Test used:     {freq_result["test_used"]}')
print(f'  Effect:        {freq_result["effect"]:+.4f}')
print(f'  95% CI:        [{freq_result["ci_lower"]:.4f}, {freq_result["ci_upper"]:.4f}]')
print(f'  p-value:       {freq_result["p_value"]:.6f}')
print(f'  Significant:   {freq_result["significant"]}')
print(f'  Cohen\'s h:     {freq_result["effect_size"]:.4f}')

In [ ]:
# Conversion rate bar chart with confidence intervals
plot_conversion_bar(
    freq_result['mean_control'], freq_result['mean_treatment'],
    freq_result['n_control'], freq_result['n_treatment'],
    experiment_id='exp_001'
)
plt.show()

## 4. Bayesian posterior computation

The Bayesian approach models each group's conversion rate with a
Beta distribution (using an uninformative Beta(1,1) prior) and
computes the posterior probability that treatment is better.

In [ ]:
bayes_result = bayesian_ab(
    control['converted'].values,
    treatment['converted'].values,
    n_samples=100000,
    seed=42
)

print('Bayesian analysis results (exp_001):')
print(f'  P(Treatment > Control): {bayes_result["prob_treatment_better"]:.3f}')
print(f'  Expected lift:          {bayes_result["expected_lift"]:.2%}')
print(f'  95% CI for lift:        [{bayes_result["lift_ci_lower"]:.2%}, {bayes_result["lift_ci_upper"]:.2%}]')
print(f'  Control posterior mean:  {bayes_result["control_mean"]:.4f}')
print(f'  Treatment posterior mean: {bayes_result["treatment_mean"]:.4f}')

In [ ]:
# Posterior distributions and lift distribution
plot_posterior_distributions(bayes_result, experiment_id='exp_001')
plt.show()

## 5. Sequential testing with O'Brien-Fleming spending function

Sequential testing allows checking results at multiple interim points
while controlling the overall Type I error rate. The O'Brien-Fleming
boundary is conservative early (requiring strong evidence to stop)
and relaxes as more data accumulates.

In [ ]:
seq_result = sequential_test(exp1, alpha=0.05, beta=0.20, n_looks=5)

print('Sequential test results (exp_001):')
print(f'  Number of looks: {seq_result["n_looks"]}')
print(f'  Early stopping:  {seq_result["stopped_early"]}')
if seq_result['stopped_early']:
    print(f'  Stopped at look: {seq_result["stop_at_look"]}')
print()

print(f'{"Look":>4s}  {"n/group":>8s}  {"Info frac":>9s}  {"Boundary":>8s}  {"Z-stat":>7s}  {"Reject":>6s}')
print('-' * 50)
for r in seq_result['results']:
    print(f'{r["look"]:4d}  {r["n_per_group"]:8,}  {r["info_fraction"]:9.3f}  '
          f'{r["z_boundary"]:8.3f}  {r["z_stat"]:7.3f}  {str(r["reject"]):>6s}')

In [ ]:
# Sequential monitoring chart
plot_sequential_monitoring(seq_result, experiment_id='exp_001')
plt.show()

## 6. Compare all three approaches on exp_001

The three methods answer different questions:
- Frequentist: "If there's no real effect, how often would we see this data?"
- Bayesian: "Given this data, how likely is it that treatment is better?"
- Sequential: "Can we stop collecting data and still trust the result?"

In [ ]:
comparison = pd.DataFrame({
    'Method': ['Frequentist', 'Bayesian', 'Sequential'],
    'Key metric': [
        f'p = {freq_result["p_value"]:.4f}',
        f'P(B>A) = {bayes_result["prob_treatment_better"]:.3f}',
        f'Stopped at look {seq_result["stop_at_look"]}' if seq_result['stopped_early']
        else 'No early stop'
    ],
    'Conclusion': [
        'Significant' if freq_result['significant'] else 'Not significant',
        'Treatment likely better' if bayes_result['prob_treatment_better'] > 0.95
        else 'Inconclusive' if bayes_result['prob_treatment_better'] > 0.5
        else 'Control likely better',
        'Reject H0' if seq_result['stopped_early'] else 'Fail to reject'
    ],
    'Effect estimate': [
        f'{freq_result["effect"]:+.4f} [{freq_result["ci_lower"]:.4f}, {freq_result["ci_upper"]:.4f}]',
        f'{bayes_result["expected_lift"]:+.2%} [{bayes_result["lift_ci_lower"]:.2%}, {bayes_result["lift_ci_upper"]:.2%}]',
        'N/A (binary decision)'
    ],
})

print('Comparison of approaches for exp_001:')
comparison

## 7. Apply all three methods to all experiments

In [ ]:
all_results = []

for exp_id in df['experiment_id'].unique():
    exp_data = df[df['experiment_id'] == exp_id]
    report = summary_report(exp_data, experiment_id=exp_id)
    print_summary(report)
    all_results.append(report)

In [ ]:
# Summary table across experiments
summary_rows = []
for r in all_results:
    fc = r['frequentist_conversion']
    b = r['bayesian']
    s = r['sequential']
    summary_rows.append({
        'Experiment': r['experiment_id'],
        'Control rate': f'{r["control_conversion_rate"]:.3%}',
        'Treatment rate': f'{r["treatment_conversion_rate"]:.3%}',
        'p-value': f'{fc["p_value"]:.4f}',
        'Freq. significant': fc['significant'],
        'P(B>A)': f'{b["prob_treatment_better"]:.3f}',
        'Early stop': s['stopped_early'],
    })

summary_df = pd.DataFrame(summary_rows).set_index('Experiment')
print('Cross-experiment summary:')
summary_df

## 8. Visualize results for all experiments

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, r in zip(axes, all_results):
    fc = r['frequentist_conversion']
    rates = [r['control_conversion_rate'], r['treatment_conversion_rate']]
    colors = ['#90CAF9', '#1565C0']
    bars = ax.bar(['Control', 'Treatment'], rates, color=colors)

    for bar, rate in zip(bars, rates):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{rate:.2%}', ha='center', va='bottom', fontsize=10)

    sig_label = 'p<0.05' if fc['significant'] else f'p={fc["p_value"]:.3f}'
    ax.set_title(f'{r["experiment_id"]} ({sig_label})')
    ax.set_ylabel('Conversion rate')

plt.suptitle('Conversion rates across experiments', fontsize=13)
plt.tight_layout()
plt.show()

## Summary

The three statistical approaches largely agree but provide different insights:

- **exp_001 (Website redesign)**: all methods detect a significant effect.
  The treatment increased conversion by ~1pp from a 3% baseline.
- **exp_002 (Pricing change)**: no method finds significance. The 0.2pp
  difference is too small to detect with n=5,000 per group.
- **exp_003 (Email subject)**: strong significance across all methods.
  The 3pp lift from 12% to 15% is both statistically and practically significant.

The **Bayesian approach** is most useful when communicating to stakeholders
("there is a 97% probability that the new design is better") while the
**sequential approach** enables faster decisions when the effect is large.

Next: meta-analysis across experiments in `04_meta_analysis.ipynb`.